# Regulatory Circular Intelligence Assistant — Colab Prototype

Runtime: set `Runtime -> Change runtime type -> T4 GPU` before running.

Pipeline: register -> parse -> chunk+embed -> relations -> status -> retrieve -> answer -> Gradio chat.

## 0. Setup — Drive, repo, dependencies, model weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/circular-intel-proto'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/raw', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/parsed', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/db', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/chroma_db', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
print('Drive folders ready at', DRIVE_ROOT)

In [ ]:
REPO_URL = 'https://github.com/dprashantit/circular-intel-proto.git'
REPO_DIR = '/content/circular-intel-proto'

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!nvcc --version 2>/dev/null || echo 'nvcc not found (normal on Colab — the prebuilt wheel below does not need it)'

**No GPU today?** If Colab says "Cannot connect to GPU backend (usage limits)", switch `Runtime -> Change runtime type` to **CPU**, then keep going — the two cells below degrade gracefully (the CUDA-wheel attempts will fail and fall back to a CPU build automatically), and the model-download cell below that lets you pick a small CPU-friendly model instead of the 7B one.

In [ ]:
# Compiling llama-cpp-python from source on Colab is unreliable (CMake/toolchain
# issues with no useful error output). Use abetlen's prebuilt CUDA wheels instead —
# try a few CUDA versions in order, falling back to a CPU-only wheel (still works,
# just slower — n_gpu_layers in llm.py will simply have no effect) if none match.
import subprocess

cuda_wheel_indices = [
    'https://abetlen.github.io/llama-cpp-python/whl/cu124',
    'https://abetlen.github.io/llama-cpp-python/whl/cu123',
    'https://abetlen.github.io/llama-cpp-python/whl/cu122',
    'https://abetlen.github.io/llama-cpp-python/whl/cu121',
]

installed = False
for idx in cuda_wheel_indices:
    print(f'Trying {idx} ...')
    result = subprocess.run(
        ['pip', 'install', '-q', 'llama-cpp-python',
         '--extra-index-url', idx, '--force-reinstall', '--no-cache-dir'],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f'Installed GPU wheel from {idx}')
        installed = True
        break
    print(result.stderr[-800:])

if not installed:
    print('No matching prebuilt CUDA wheel found — falling back to CPU-only build.')
    print('Answers will still work, just slower per token.')
    subprocess.run(['pip', 'install', '-q', 'llama-cpp-python', '--force-reinstall', '--no-cache-dir'])

In [ ]:
# MODEL_SIZE toggle:
#  'small' — Qwen2.5-1.5B-Instruct, ~1GB, runs tolerably on CPU. Use this when
#            you don't have GPU access, to validate pipeline logic end-to-end
#            (relation typing, answer composition) even if answer quality is
#            weaker and generation is slow.
#  'large' — Qwen2.5-7B-Instruct, ~4.5GB, needs GPU offload to be usably fast.
#            Switch back to this once T4 access returns.
MODEL_SIZE = 'small'  # 'small' or 'large'

MODEL_CONFIGS = {
    'small': {
        'filename': 'qwen2.5-1.5b-instruct-q4_k_m.gguf',
        'url': 'https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf',
    },
    'large': {
        'filename': 'qwen2.5-7b-instruct-q4_k_m.gguf',
        'url': 'https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m.gguf',
    },
}

cfg = MODEL_CONFIGS[MODEL_SIZE]
MODEL_PATH = f"{DRIVE_ROOT}/models/{cfg['filename']}"
MODEL_URL = cfg['url']

import os
if not os.path.exists(MODEL_PATH):
    !wget -O "{MODEL_PATH}" "{MODEL_URL}"
else:
    print('Model already downloaded on Drive, skipping.')

print('Using model:', MODEL_PATH)

In [ ]:
MODEL_PATH = f'{DRIVE_ROOT}/models/qwen2.5-7b-instruct-q4_k_m.gguf'
MODEL_URL = 'https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m.gguf'

import os
if not os.path.exists(MODEL_PATH):
    !wget -O "{MODEL_PATH}" "{MODEL_URL}"
else:
    print('Model already downloaded on Drive, skipping.')

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src import db, parse, chunk as chunk_mod, embed, relations, retrieve, answer, llm

DB_PATH = f'{DRIVE_ROOT}/db/register.sqlite'
CHROMA_DIR = f'{DRIVE_ROOT}/chroma_db'
RAW_DIR = f'{DRIVE_ROOT}/data/raw'
PARSED_DIR = f'{DRIVE_ROOT}/data/parsed'
SOURCES_CFG_PATH = f'{REPO_DIR}/config/sources.yaml'

db.init_db(DB_PATH)
sources_cfg = parse.load_sources_config(SOURCES_CFG_PATH)
print('DB initialized at', DB_PATH)

## 0b. (Optional) Unzip your sample circulars into Drive
Upload `circulars_filtered.zip` to Colab's file panel first, or to Drive directly, then point `ZIP_PATH` at it.
Files should end up sorted by source — this cell sorts by filename prefix (`NSE_...`, `BSE_...`).

In [ ]:
ZIP_PATH = '/content/circulars_filtered.zip'  # upload via Colab's file panel, or point at a Drive path

import zipfile, shutil, glob
if os.path.exists(ZIP_PATH):
    tmp_dir = '/content/_unzipped'
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(tmp_dir)
    for f in glob.glob(f'{tmp_dir}/**/*.pdf', recursive=True):
        fname = os.path.basename(f)
        source = parse.detect_source(fname, sources_cfg) or 'UNKNOWN'
        dest_dir = f'{RAW_DIR}/{source}'
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(f, f'{dest_dir}/{fname}')
    print('Sorted into', RAW_DIR)
else:
    print('No zip found at', ZIP_PATH, '— skip if files are already in data/raw/')

## 1. Register + parse

In [ ]:
import glob
from tqdm.auto import tqdm

pdf_paths = sorted(glob.glob(f'{RAW_DIR}/**/*.pdf', recursive=True))
print(f'{len(pdf_paths)} PDFs found')

parsed_results = []
with db.connect(DB_PATH) as conn:
    for pdf_path in tqdm(pdf_paths):
        result = parse.parse_and_save(pdf_path, sources_cfg, PARSED_DIR)
        if not result['source']:
            print('Unrecognized source, skipping:', pdf_path)
            continue
        circular_id = db.upsert_circular(
            conn,
            source=result['source'],
            native_id=result.get('native_id', result['filename']),
            official_ref=result.get('official_ref'),
            issue_date=result.get('issue_date') or '1900-01-01',
            title=result.get('title'),
            department=result.get('department'),
            category=result.get('category'),
            segment=result.get('segment'),
            raw_path=pdf_path,
            parsed_path=result['parsed_path'],
        )
        result['circular_id'] = circular_id
        parsed_results.append(result)

print(f'Registered {len(parsed_results)} circulars. Quality issues:',
      sum(1 for r in parsed_results if r['quality'] != 'ok'))

## 2. Chunk + embed

In [ ]:
with db.connect(DB_PATH) as conn:
    for result in tqdm(parsed_results):
        header = chunk_mod.build_header(
            result['source'], result.get('native_id', ''), result.get('issue_date', ''), result.get('title')
        )
        chunks = chunk_mod.chunk_text(result['text'], header)
        if not chunks:
            continue

        chunk_ids = [f"{result['circular_id']}_{c.ordinal}" for c in chunks]
        texts = [c.context_text for c in chunks]
        metadatas = [
            {'circular_id': result['circular_id'], 'source': result['source'],
             'native_id': result.get('native_id', ''), 'issue_date': result.get('issue_date', '')}
            for _ in chunks
        ]
        embed.upsert_chunks(CHROMA_DIR, chunk_ids, texts, metadatas)
        for cid, c in zip(chunk_ids, chunks):
            db.insert_chunk_meta(conn, cid, result['circular_id'], c.ordinal, c.section_path, c.text[:200])

print('Chunking + embedding done.')

import subprocess
has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
n_gpu_layers = -1 if has_gpu else 0  # -1 = offload all layers; 0 = CPU only
print('GPU detected:' , has_gpu, '-> n_gpu_layers =', n_gpu_layers)

llm.load_model(MODEL_PATH, n_gpu_layers=n_gpu_layers, n_ctx=8192)
print(llm.chat('You are terse.', 'Say OK if you are working.', max_tokens=10))

In [ ]:
llm.load_model(MODEL_PATH, n_gpu_layers=-1, n_ctx=8192)
print(llm.chat('You are terse.', 'Say OK if you are working.', max_tokens=10))

## 4. Relation extraction + status computation

In [ ]:
with db.connect(DB_PATH) as conn:
    total_relations = 0
    for result in tqdm(parsed_results):
        n = relations.extract_relations_for_circular(
            conn, result['circular_id'], result['source'], result['text'], sources_cfg
        )
        total_relations += n
    print(f'{total_relations} candidate citations found')

    for result in parsed_results:
        relations.compute_status(conn, result['circular_id'])

    rows = conn.execute('SELECT source, native_id, status, status_reason FROM circular').fetchall()
    for r in rows:
        if r['status'] != 'in_force':
            print(dict(r))

## 5. Test retrieval + answer composition

In [ ]:
question = 'What are the mock trading timings for the Currency Derivatives segment?'
hits = retrieve.retrieve(DB_PATH, CHROMA_DIR, question, top_k=8)
result = answer.compose_answer(question, hits, DB_PATH)
print(result['answer'])
print('\nCitations:', [c['label'] for c in result['citations']])
print('Unsupported citation marks (should be empty):', result['unsupported'])

## 6. Gradio chat UI

In [ ]:
import gradio as gr
import time

def respond(message, history):
    t0 = time.time()
    hits = retrieve.retrieve(DB_PATH, CHROMA_DIR, message, top_k=8)
    result = answer.compose_answer(message, hits, DB_PATH)
    latency_ms = int((time.time() - t0) * 1000)
    with db.connect(DB_PATH) as conn:
        db.log_query(conn, message, [h['chunk_id'] for h in hits], result['answer'], latency_ms)
    return result['answer']

demo = gr.ChatInterface(
    respond,
    title='Regulatory Circular Assistant (prototype)',
    description='Answers are generated only from the ingested circulars, with citations.',
)
demo.launch(share=True)